# 04 — Predykcja: GP Singapuru 2026

**Cel notebooka:** wytrenować finalne modele na **wszystkich** zakończonych wyścigach 2026 i przewidzieć dzisiejszy wyścig: zwycięzcę, podium, top 10 i najszybsze okrążenie.

W `03_model` walidacja odpowiadała na pytanie *„czy to podejście działa?”*. Tu już nie walidujemy — bierzemy całą dostępną wiedzę i robimy jedną prognozę.

## 4.0 Wczytanie cech

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression

data = pd.read_csv("data/features.csv")
races = pd.read_csv("data/races_2026.csv")
feature_lists = json.load(open("data/feature_list.json"))
FEATURES, PACE_FEATURES = feature_lists["all"], feature_lists["pace"]

TARGET_ROUND = races.loc[races["country_name"] == "Singapore", "round"].item()
FINISHED_ROUNDS = races.loc[races["finished"] & (races["round"] != TARGET_ROUND), "round"].tolist()

train_data = data[data["round"].isin(FINISHED_ROUNDS) & (data["round"] >= 2)]
target = data[data["round"] == TARGET_ROUND].copy()
print(f"Trening: {len(train_data)} wierszy z {train_data['round'].nunique()} wyścigów | Singapur: {len(target)} kierowców")

Trening: 328 wierszy z 15 wyścigów | Singapur: 22 kierowców


**Co tu się dzieje:**
- `FINISHED_ROUNDS` celowo **wyklucza Singapur**, nawet gdy już się odbędzie i uruchomisz notebooki ponownie. Dzięki temu prognoza po wyścigu jest odtwarzalna — model nie „widzi” wyniku, który przewiduje.
- `target` — 22 wiersze Singapuru: cechy znane w sobotę wieczorem, `finish` pusty.

## 4.1 Kolejność na mecie

**Co robimy:** trenujemy **model tempa** z `03_model` (3.5b) — `Ridge` uczony na kierowcach, którzy dojechali do mety — na wszystkich zakończonych wyścigach i przewidujemy pozycję każdego kierowcy w Singapurze.

**Dlaczego ten model:** w walidacji walk-forward jako jedyny pobił baseline w typowaniu zwycięzcy, nie tracąc na podium i top 10. Jest prosty, odporny na małą liczbę danych, a jego wagi da się zinterpretować.

In [2]:
finishers = train_data[~train_data["retired"]]
position_model = make_pipeline(StandardScaler(), Ridge(alpha=10.0))
position_model.fit(finishers[PACE_FEATURES], finishers["finish"])

target["predicted_finish"] = position_model.predict(target[PACE_FEATURES])
target = target.sort_values("predicted_finish").reset_index(drop=True)
target["predicted_rank"] = target.index + 1

target[["predicted_rank", "name_acronym", "team_name", "grid", "sprint_pos", "predicted_finish",
        "driver_form", "team_dnf_rate", "driver_skill"]].round(2)

,predicted_rank,name_acronym,team_name,grid,sprint_pos,predicted_finish,driver_form,team_dnf_rate,driver_skill
0,1,VER,Red Bull Racing,1.0,1.0,2.27,1.67,0.22,-0.60
1,2,LEC,Ferrari,2.0,3.0,3.04,4.00,0.12,-0.14
2,3,ANT,Mercedes,4.0,4.0,3.45,2.67,0.12,0.38
3,4,HAM,Ferrari,3.0,2.0,3.97,10.33,0.12,0.15
4,5,RUS,Mercedes,6.0,6.0,4.78,8.67,0.12,-0.23
5,6,NOR,McLaren,5.0,8.0,5.94,11.33,0.22,-0.11
6,7,PIA,McLaren,7.0,14.0,7.41,9.00,0.22,0.11
7,8,LAW,Racing Bulls,9.0,5.0,7.60,8.33,0.06,0.27
8,9,GAS,Alpine,8.0,9.0,8.53,16.67,0.09,-0.20
9,10,HUL,Audi,11.0,7.0,9.10,10.67,0.16,-0.15


**Co tu się dzieje:**
- `predicted_finish` to przewidywana pozycja **pod warunkiem, że kierowca dojedzie** — liczba ułamkowa, np. 3,4. Ważna jest kolejność, a nie sama wartość, więc sortujemy i nadajemy rangi 1–22.
- Model nie uwzględnia ryzyka awarii — `team_dnf_rate` jest w tabeli tylko do wglądu. Faworyt z zespołu z wysokim `team_dnf_rate` jest pewniakiem mniej, niż sugeruje sama kolejność.
- Porównaj `predicted_rank` z `grid`: gdzie model się z nim nie zgadza i która cecha za tym stoi?

## 4.2 Najszybsze okrążenie

In [3]:
fl_model = make_pipeline(StandardScaler(), LogisticRegression(C=0.3))
fl_model.fit(train_data[FEATURES], train_data["fastest_lap"])

proba = fl_model.predict_proba(target[FEATURES])[:, 1]
target["fastest_lap_pct"] = proba / proba.sum() * 100
target.sort_values("fastest_lap_pct", ascending=False)[["name_acronym", "team_name", "grid", "fastest_lap_pct"]].head(5).round(1)

,name_acronym,team_name,grid,fastest_lap_pct
2,ANT,Mercedes,4.0,33.7
0,VER,Red Bull Racing,1.0,19.3
1,LEC,Ferrari,2.0,18.0
3,HAM,Ferrari,3.0,10.5
4,RUS,Mercedes,6.0,7.2


**Co tu się dzieje:**
- Regresja logistyczna daje każdemu kierowcy osobne prawdopodobieństwo, które nie sumują się do 1. Ale w wyścigu najszybsze okrążenie ma **dokładnie jeden** kierowca — więc normalizujemy: `proba / proba.sum()`. Teraz to udział „szans” w tym konkretnym wyścigu.
- Te procenty traktuj orientacyjnie — model uczył się na ~15 przykładach pozytywnych.

## 4.3 Prognoza

In [4]:
names = target["name_acronym"].tolist()
fl_pick = target.loc[target["fastest_lap_pct"].idxmax()]

print("🏁 GP SINGAPURU 2026 — PROGNOZA")
print(f"Zwycięzca:            {names[0]}")
print(f"Podium:               {', '.join(names[:3])}")
print(f"Top 10:               {', '.join(names[:10])}")
print(f"Najszybsze okrążenie: {fl_pick['name_acronym']} ({fl_pick['fastest_lap_pct']:.0f}%)")

🏁 GP SINGAPURU 2026 — PROGNOZA
Zwycięzca:            VER
Podium:               VER, LEC, ANT
Top 10:               VER, LEC, ANT, HAM, RUS, NOR, PIA, LAW, GAS, HUL
Najszybsze okrążenie: ANT (34%)


## 4.4 Zamrożenie prognozy

**Co robimy:** zapisujemy prognozę do pliku z godziną utworzenia.

**Dlaczego:** po wyścigu chcemy porównać prognozę z rzeczywistością. Gdybyśmy ją po prostu przeliczyli, dane i modele mogłyby się zmienić w międzyczasie. Zapisany plik to **dowód, co przewidzieliśmy przed startem**. Komórka nie nadpisuje istniejącego pliku.

In [5]:
PRED_PATH = Path("data/prediction_singapore_2026.csv")
if PRED_PATH.exists():
    print("Prognoza już zamrożona:", PRED_PATH, "— nie nadpisuję.")
else:
    frozen = target[["predicted_rank", "name_acronym", "team_name", "grid", "predicted_finish", "fastest_lap_pct"]].copy()
    frozen["created_at"] = pd.Timestamp.now(tz="UTC")
    frozen.to_csv(PRED_PATH, index=False)
    print("Zapisano:", PRED_PATH)

Prognoza już zamrożona: data/prediction_singapore_2026.csv — nie nadpisuję.


## 4.5 Po wyścigu: prognoza vs rzeczywistość

**Jak uruchomić (po ok. 16:00):**
1. W `01_dane` uruchom wszystkie komórki ponownie — Singapur będzie już oznaczony jako zakończony, a `get()` pobierze wynik (pusty plik nie był zapisywany, więc nie trzeba niczego kasować).
2. Uruchom poniższą komórkę.

Porównujemy z **zamrożoną** prognozą tymi samymi miarami co w walidacji.

In [6]:
frozen = pd.read_csv(PRED_PATH)
results = pd.read_csv("data/results_2026.csv")
fastest_laps = pd.read_csv("data/fastest_laps_2026.csv")
actual = results[results["round"] == TARGET_ROUND].copy()

if actual.empty:
    print("Wyniku Singapuru jeszcze nie ma — uruchom ponownie 01_dane po wyścigu.")
else:
    actual["finish"] = actual["position"].fillna(22)
    actual = actual.sort_values("finish")
    pred_names = frozen.sort_values("predicted_rank")["name_acronym"].tolist()
    real_names = actual["name_acronym"].tolist()
    real_fl = fastest_laps.loc[fastest_laps["round"] == TARGET_ROUND, "name_acronym"].item()
    pred_fl = frozen.loc[frozen["fastest_lap_pct"].idxmax(), "name_acronym"]

    print(f"Zwycięzca:  przewidziany {pred_names[0]}, faktyczny {real_names[0]}  {'✅' if pred_names[0] == real_names[0] else '❌'}")
    print(f"Podium:     trafione {len(set(pred_names[:3]) & set(real_names[:3]))}/3")
    print(f"Top 10:     trafione {len(set(pred_names[:10]) & set(real_names[:10]))}/10")
    print(f"Najszybsze: przewidziany {pred_fl}, faktyczny {real_fl}  {'✅' if pred_fl == real_fl else '❌'}")

    comparison = frozen.merge(actual[["name_acronym", "finish", "dnf"]], on="name_acronym", how="left")
    comparison["error"] = comparison["finish"] - comparison["predicted_rank"]
    display(comparison[["predicted_rank", "name_acronym", "team_name", "grid", "finish", "error", "dnf"]])

Zwycięzca:  przewidziany VER, faktyczny VER  ✅
Podium:     trafione 3/3
Top 10:     trafione 9/10
Najszybsze: przewidziany ANT, faktyczny LEC  ❌


,predicted_rank,name_acronym,team_name,grid,finish,error,dnf
0,1,VER,Red Bull Racing,1.0,1.0,0.0,False
1,2,LEC,Ferrari,2.0,2.0,0.0,False
2,3,ANT,Mercedes,4.0,3.0,0.0,False
3,4,HAM,Ferrari,3.0,6.0,2.0,False
4,5,RUS,Mercedes,6.0,9.0,4.0,False
5,6,NOR,McLaren,5.0,4.0,-2.0,False
6,7,PIA,McLaren,7.0,5.0,-2.0,False
7,8,LAW,Racing Bulls,9.0,7.0,-1.0,False
8,9,GAS,Alpine,8.0,11.0,2.0,False
9,10,HUL,Audi,11.0,8.0,-2.0,False


**Co tu się dzieje:**
- `error` = faktyczna pozycja − przewidziana. Duże dodatnie = model przecenił kierowcę, duże ujemne = nie docenił.
- To punkt wyjścia do **analizy błędów**: czy pomyłki to awarie (nieprzewidywalne), samochód bezpieczeństwa, strategia — czy coś, co model mógł wiedzieć, a nie wiedział? Te ostatnie to pomysły na nowe cechy.